# Colab GPU Smoke Test

This notebook implements the intended workflow:

```text
Codex in VS Code edits local repo
        ↓
push changes to GitHub
        ↓
Google Colab pulls/clones latest GitHub code
        ↓
Colab runs the code on GPU
```

Use **Runtime > Change runtime type > GPU**, then run the cells from top to bottom.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone or Pull Latest GitHub Code

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
print("repo root:", ROOT)
print("commit:", commit)

## 3. Install/Verify Dependencies

Colab often already has NumPy/SciPy/Matplotlib and CuPy. If CuPy is installed for the first time here, restart the runtime and rerun from the top.

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")

print("Dependencies are ready.")

## 4. Verify CuPy Backend

In [ ]:
!python scripts/check_backend.py

In [ ]:
from aberration_simulation.backend import HAS_CUPY, xp

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU smoke path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())

## 5. Run Smoke Test

In [ ]:
!python scripts/run_smoke_test.py

## 6. Generate Line-Profile Plots

In [ ]:
!python scripts/plot_line_profiles.py

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for path in sorted(Path("outputs/plots").glob("line_profiles_*.png")):
    print(path)
    display(Image(filename=str(path)))